In [173]:
import os
from pathlib import Path
import warnings

import pandas as pd
import numpy as np

import re

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize, sent_tokenize, TweetTokenizer, MWETokenizer
from nltk.probability import FreqDist

from gensim.models import Word2Vec

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer", "attribute_ruler"])

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

warnings.filterwarnings("ignore")

[nltk_data] Downloading package punkt to /Users/nicole/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nicole/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/nicole/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [111]:
linkedin = pd.read_csv('data/influencers_data.csv')
linkedin_data = linkedin.dropna(subset=['content'])['content']

In [ ]:
# tokenizer = TweetTokenizer()

# initial_tokens = tokenizer.tokenize(linkedin_data[0])

# li_tokens = linkedin_data.map(lambda x: tokenizer.tokenize(x))

# li_tokens = li_tokens.reset_index(drop=True)

# li_tokens = [[i for i in lst if any(c.isalnum() for c in i)] for lst in li_tokens]

# li_tokens = [nltk.pos_tag(lst) for lst in li_tokens]
# li_tokens = [[i for i, tag in lst if tag not in ('NNP', 'NNPS')] for lst in li_tokens]

In [176]:
texts = linkedin_data.dropna().astype(str).tolist()

li_tokens = []
for doc in nlp.pipe(texts, batch_size=50, n_process=1):
    tokens = [
        token.text for token in doc
        if token.pos_ != "PROPN" and any(c.isalnum() for c in token.text)
    ]
    li_tokens.append(tokens)

In [129]:
def print_word_freq(tokens, target_words):
    if type(tokens[0]) == list:
        unpacked_tokens = [i for lst in tokens for i in lst]
        # print(len(unpacked_tokens))
        fdist = FreqDist(unpacked_tokens)
    else:
        # print(len(tokens))
        fdist = FreqDist(tokens)
    value_counts = {word : fdist[word] for word in target_words}
    value_counts['total_tokens'] = fdist.N()
    return value_counts

In [177]:
targets = ['build', 'ship', 'wonderful', 'business', 'fellow', 'friend']
li_target_freq = print_word_freq(li_tokens, targets)
li_target_freq

{'build': 765,
 'ship': 49,
 'wonderful': 242,
 'business': 2299,
 'fellow': 101,
 'friend': 358,
 'total_tokens': 1683588}

In [105]:
def find_params(tokens):
    if type(tokens[0]) == list:
        unpacked_tokens = [i for lst in tokens for i in lst]
        tokens_count = len(unpacked_tokens)
    else:
        tokens_count = len(tokens)

    model_params = {
        'tiny' : {
            'vector_size' : 25,
            'min_count' : 3,
            'epochs' : 30,
            'negative' : 15
        },
        'small': {
            'vector_size' : 50,
            'min_count' : 7,
            'epochs' : 20,
            'negative' : 10
        },
        'medium' : {
            'vector_size' : 100,
            'min_count' : 10,
            'epochs' : 10,
            'negative' : 5
        },
        'large' : {
            'vector_size' : 100,
            'min_count' : 10,
            'epochs' : 5,
            'negative' : 5
        }
    }

    if tokens_count < 500000:
        model_size = 'tiny'
    elif tokens_count < 5000000:
        model_size = 'small'
    elif tokens_count < 10000000:
        model_size = 'medium'
    else:
        model_size = 'large'

    return model_params[model_size]

In [ ]:
# def create_coha_tokens(decade):
#     dir_path = Path(f'data/coha-samples-text/{decade}')
#     all_tokens = []

#     for f in dir_path.iterdir():
#         if f.is_file():
#             try:
#                 with open(f, 'r', encoding='utf-8') as file:
#                     content = file.readlines()
#                     sentences = sent_tokenize(content[-1].lower().title())
#                     sentences = [word_tokenize(sent) for sent in sentences]
#                     sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
#                     sentences = [nltk.pos_tag(lst) for lst in sentences]
#                     print(sentences)
#                     sentences = [[i[0] for i in lst] for lst in sentences]
#                     sentences = [[i for i, tag in lst if tag not in ('NNP', 'NNPS')] for lst in sentences]
#                     all_tokens.extend(sentences)
#             except Exception as e:
#                 print(f'Could not read {f.name}: {e}')
    
#     return all_tokens

In [174]:
def create_coha_tokens(decade, nlp):
    dir_path = Path(f'data/coha-samples-text/{decade}')
    all_tokens = []

    files = [f for f in dir_path.iterdir() if f.is_file()]
    texts = []
    valid_files = []

    for f in files:
        try:
            with open(f, 'r', encoding='utf-8') as file:
                content = file.readlines()
                texts.append(content[-1])
                valid_files.append(f)
        except Exception as e:
            print(f'Could not read {f.name}: {e}')

    for doc in nlp.pipe(texts, batch_size=50):
        for sent in doc.sents:
            tokens = [
                token.text for token in sent
                if token.pos_ != "PROPN" and any(c.isalnum() for c in token.text)
            ]
            all_tokens.append(tokens)

    return all_tokens

In [172]:
create_coha_tokens('dummy', nlp)

[['The',
  'author',
  'is',
  'indebted',
  'to',
  'one',
  'of',
  'the',
  'novels',
  'of',
  'Le',
  'Brun',
  'for',
  'the',
  'ground',
  'work',
  'of',
  'this',
  'little',
  'comedy'],
 ['DRAMATIS', 'PERSON'],
 ['Philadelphia',
  'Count',
  'Almeyda',
  'Mr.',
  'Robertson',
  'Count',
  'Arandez',
  'Warren',
  'Carlos',
  'Wood',
  'Pacomo',
  'Jefferson',
  'Gusman',
  'Abercrombie',
  'Pedrillo',
  'Durang',
  'Herald',
  'Jackson',
  'Eugenia',
  'Mrs.',
  'Entwistle',
  'Beatrice',
  'Francis',
  'Flora',
  'Claude',
  'Ladies',
  'knights',
  'men',
  'at',
  'arms',
  'pages',
  'servants'],
 ['SCENE',
  'at',
  'the',
  'castle',
  'of',
  'count',
  'Almeyda',
  'in',
  'Catalonia',
  'during',
  'the',
  'thirteenth',
  'century'],
 ['Time', 'twenty', 'four', 'hours'],
 ['Main', 'text', 'ACT', 'I.', 'SCENE', 'I', 'a', 'thick', 'wood'],
 ['Carlos'],
 ['Carlos', 'entering', 'Come', 'along', 'Pacomo'],
 ['Pacomo'],
 ['Pacomo', 'without'],
 ['Here',
  'I',
  'come',

In [ ]:
# def create_coca_tokens(fp):
#     dir_path = Path(fp)
#     all_tokens = []

#     for f in dir_path.iterdir():
#         if f.is_file():
#             try:
#                 with open(f, 'r', encoding='utf-8') as file:
#                     content = file.readlines()
#                     sentences = [re.sub(r'@@\d+', '', s) for s in content]
#                     sentences = [sent_tokenize(s) for s in sentences]
#                     sentences = [word_tokenize(s) for sent in sentences for s in sent]
#                     sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
#                     all_tokens.extend(sentences)
#             except Exception as e:
#                 print(f'Could not read {f.name}: {e}')
    
#     return all_tokens

In [175]:
def create_coca_tokens(fp, nlp):
    dir_path = Path(fp)
    all_tokens = []

    texts = []
    for f in dir_path.iterdir():
        if f.is_file():
            try:
                with open(f, 'r', encoding='utf-8') as file:
                    content = file.readlines()
                    cleaned = [re.sub(r'@@\d+', '', line) for line in content]
                    texts.extend(cleaned)  # each line becomes its own text to parse
            except Exception as e:
                print(f'Could not read {f.name}: {e}')

    for doc in nlp.pipe(texts, batch_size=50, n_process=1):
        for sent in doc.sents:
            tokens = [
                token.text for token in sent
                if token.pos_ != "PROPN" and any(c.isalnum() for c in token.text)
            ]
            all_tokens.append(tokens)

    return all_tokens

In [69]:
def create_model(tokens, params):
    model = Word2Vec(
        sentences=tokens, 
        vector_size=params['vector_size'],
        window=5,            
        min_count=params['min_count'],     
        workers=4,
        sg=1,
        epochs=params['epochs'],
        negative=params['negative']
    )
    return model

In [116]:
li_params = find_params(li_tokens)
li_params

{'vector_size': 50, 'min_count': 7, 'epochs': 20, 'negative': 10}

In [117]:
li_model = create_model(li_tokens, li_params)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [101]:
def find_neighbors(model, targets):
    # structure:
    # {target_word : [(neighbor1, cos_similarity), (neighbor2, cos_similarity)]}
    neighbors = {
        i : model.wv.most_similar(i) for i in targets
    }
    for n in neighbors:
        print(f"neighbors of '{n}': ")
        print([i[0] for i in neighbors[n]])
    return neighbors

In [118]:
li_neighborhoods = find_neighbors(li_model, targets)

neighbors of 'build': 
['develop', 'create', 'building', 'grow', 'brand', 'leveraged', 'hone', 'generate', 'reputation', 'drive']
neighbors of 'ship': 
['cruise', 'plane', 'debut', 'fly', 'vibe', 'delicious', 'rent', 'breakfast', 'trains', 'eat']
neighbors of 'wonderful': 
['lovely', 'amazing', 'incredible', 'great', 'terrific', 'fantastic', 'fabulous', 'beautiful', 'well-deserved', 'Lovely']
neighbors of 'business': 
['company', 'owners', 'strategy', 'brand', 'model', 'role', 'firm', 'fundable', 'operations', 'industry']
neighbors of 'fellow': 
['hometown', 'proud', 'Proud', 'Huge', 'community', 'Humbled', 'Wonderful', 'Mentors', 'footsteps', 'Lovely']
neighbors of 'friend': 
['colleague', 'dude', 'daughter', 'wife', 'dad', 'boss', 'mentor', 'mom', 'sister', 'accountant']


In [163]:
def coha_word_neighborhoods(decades, targets):
    tokens = []
    print('Tokenizing data...')
    for i in decades:
        tokens.extend(create_coha_tokens(i))
    print('Tokens created.')
    freq_dist = print_word_freq(tokens, targets)
    num_tokens = freq_dist['total_tokens']
    freq_dist = {t : freq_dist[t] for t in targets}
    print(f'The frequency distribution of target words is: {freq_dist}')
    params = find_params(tokens)
    
    coha_model = create_model(tokens, params)
    return find_neighbors(coha_model, targets)

In [165]:
pre1850_tokens = create_coha_tokens('pre-1850')
print_word_freq(pre1850_tokens, targets)

[['The', 'author', 'is', 'indebted', 'to', 'one', 'of', 'the', 'novels', 'of', 'for', 'the', 'ground', 'work', 'of', 'this', 'little', 'comedy'], ['PERSON'], ['knights', 'men', 'at', 'arms', 'pages', 'servants'], ['at', 'the', 'castle', 'of', 'count', 'in', 'during', 'the', 'thirteenth', 'century'], ['Time', 'twenty', 'four', 'hours'], ['Main', 'text', 'I', 'a', 'thick', 'wood'], [], ['entering', 'Come', 'along'], [], ['without'], ['Here', 'I', 'come', 'signor', 'as', 'fast', 'as', 'the', 'briers', 'will', 'permit', 'me'], ['Pacomo', 'enters', 'dragging', 'a', 'large', 'portmanteau', 'he', 'stops', 'at', 'the', 'wing', 'looking', 'ruefully', 'around'], ['No', 'outlet', 'yet', 'sir'], ['Car'], [], ['Pac'], ['resting', 'his', 'portmanteau', 'against', 'a', 'tree', 'and', 'taking', 'his', 'seat', 'upon', 'it'], ['Why', 'then', 'sir', 'we', 'might', 'as', 'well', 'make', 'up', 'our', 'minds', 'to', 'starve'], ['This', 'is', 'a', 'pleasant', 'adventure'], ['Pac'], ['You', 'think', 'so'], ['

{'build': 5,
 'ship': 110,
 'wonderful': 10,
 'business': 117,
 'fellow': 114,
 'friend': 181,
 'total_tokens': 462438}

In [168]:
coha_word_neighborhoods(['dummy'], targets=['love', 'history'])

Tokenizing data...
[['The', 'author', 'is', 'indebted', 'to', 'one', 'of', 'the', 'novels', 'of', 'for', 'the', 'ground', 'work', 'of', 'this', 'little', 'comedy'], ['PERSON'], ['knights', 'men', 'at', 'arms', 'pages', 'servants'], ['at', 'the', 'castle', 'of', 'count', 'in', 'during', 'the', 'thirteenth', 'century'], ['Time', 'twenty', 'four', 'hours'], ['Main', 'text', 'I', 'a', 'thick', 'wood'], [], ['entering', 'Come', 'along'], [], ['without'], ['Here', 'I', 'come', 'signor', 'as', 'fast', 'as', 'the', 'briers', 'will', 'permit', 'me'], ['Pacomo', 'enters', 'dragging', 'a', 'large', 'portmanteau', 'he', 'stops', 'at', 'the', 'wing', 'looking', 'ruefully', 'around'], ['No', 'outlet', 'yet', 'sir'], ['Car'], [], ['Pac'], ['resting', 'his', 'portmanteau', 'against', 'a', 'tree', 'and', 'taking', 'his', 'seat', 'upon', 'it'], ['Why', 'then', 'sir', 'we', 'might', 'as', 'well', 'make', 'up', 'our', 'minds', 'to', 'starve'], ['This', 'is', 'a', 'pleasant', 'adventure'], ['Pac'], ['You',

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


neighbors of 'love': 
['sweet', 'undefiled', 'link', 'compare', '12', 'skies', 'parting', 'For', 'saintly', 'desolate']
neighbors of 'history': 
['fund', 'recorded', 'chief', 'projects', 'revolutionary', 'romance', 'ordinary', 'quality', 'uncertain', 'liberties']


{'love': [('sweet', 0.8794199228286743),
  ('undefiled', 0.8539538979530334),
  ('link', 0.8535720705986023),
  ('compare', 0.8386640548706055),
  ('12', 0.8354005217552185),
  ('skies', 0.8352183103561401),
  ('parting', 0.8320534825325012),
  ('For', 0.8307139873504639),
  ('saintly', 0.8287144303321838),
  ('desolate', 0.8266133069992065)],
 'history': [('fund', 0.8621144890785217),
  ('recorded', 0.8557971715927124),
  ('chief', 0.8497155904769897),
  ('projects', 0.8318289518356323),
  ('revolutionary', 0.8289915323257446),
  ('romance', 0.8220425248146057),
  ('ordinary', 0.8201590776443481),
  ('quality', 0.806337296962738),
  ('uncertain', 0.8048117756843567),
  ('liberties', 0.8039273619651794)]}

In [ ]:
print_word_freq(d1900_tokens, targets)
print_word_freq(d1910_tokens, targets)
print_word_freq(d1920_tokens, targets)
print_word_freq(d1930_tokens, targets)
print_word_freq(d1940_tokens, targets)
print_word_freq(d1950_tokens, targets)
print_word_freq(d1960_tokens, targets)
print_word_freq(d1970_tokens, targets)
print_word_freq(d1980_tokens, targets)
print_word_freq(d1990_tokens, targets)
print_word_freq(d2000_tokens, targets)

234177
{'build': 23, 'ship': 46, 'wonderful': 19, 'business': 143, 'fellow': 99, 'friend': 97}
195147
{'build': 3, 'ship': 44, 'wonderful': 26, 'business': 54, 'fellow': 15, 'friend': 32}
223336
{'build': 14, 'ship': 31, 'wonderful': 21, 'business': 132, 'fellow': 16, 'friend': 28}
195847
{'build': 13, 'ship': 15, 'wonderful': 1, 'business': 91, 'fellow': 12, 'friend': 16}
322316
{'build': 10, 'ship': 160, 'wonderful': 9, 'business': 72, 'fellow': 22, 'friend': 52}
128604
{'build': 9, 'ship': 2, 'wonderful': 13, 'business': 52, 'fellow': 14, 'friend': 25}
250938
{'build': 19, 'ship': 42, 'wonderful': 10, 'business': 83, 'fellow': 17, 'friend': 30}
326300
{'build': 9, 'ship': 3, 'wonderful': 6, 'business': 44, 'fellow': 16, 'friend': 81}
199256
{'build': 13, 'ship': 28, 'wonderful': 5, 'business': 69, 'fellow': 12, 'friend': 34}
224495
{'build': 42, 'ship': 4, 'wonderful': 13, 'business': 51, 'fellow': 5, 'friend': 23}
247468
{'build': 18, 'ship': 24, 'wonderful': 12, 'business': 60, 'f

{'build': 18,
 'ship': 24,
 'wonderful': 12,
 'business': 60,
 'fellow': 15,
 'friend': 43}

In [ ]:
pre1900_model = Word2Vec(
    sentences=pre1900_tokens, 
    vector_size=50,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    epochs=20,
    negative=10
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [165]:
pre1900_model.wv.most_similar('build', topn=10)

[('feed', 0.7074720859527588),
 ('adventurers', 0.6827496886253357),
 ('selecting', 0.6532171964645386),
 ('protect', 0.6415084600448608),
 ('yourselves', 0.6414114832878113),
 ('farms', 0.6333451867103577),
 ('hire', 0.6291592121124268),
 ('whites', 0.6203471422195435),
 ('furnaces', 0.619622528553009),
 ('strike', 0.6181607842445374)]

In [188]:
coca_tokens = create_coca_tokens('data/coca-samples-text/')

In [189]:
sum([len(s) for s in coca_tokens])

9378210

In [ ]:
coca_model = Word2Vec(
    sentences=coca_tokens,
    vector_size=100,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    epochs=10,
    negative=5
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [193]:
coca_model.wv.most_similar('build', topn=10)

[('create', 0.7406520247459412),
 ('develop', 0.7308622598648071),
 ('innovate', 0.692916989326477),
 ('modernize', 0.6891975998878479),
 ('rebuild', 0.649296760559082),
 ('forge', 0.6476582288742065),
 ('allocate', 0.646988034248352),
 ('reconfigure', 0.6385540962219238),
 ('dismantle', 0.637188196182251),
 ('reconstruct', 0.6347952485084534)]

In [203]:
def create_enron_tokens(fp):
    dir_path = Path(fp)
    all_tokens = []

    for f in dir_path.iterdir():
        if f.is_file():
            try:
                with open(f, 'r', encoding='utf-8') as file:
                    content = file.readlines()
                    sentences = ''.join(content).replace('\n', ' ')
                    sentences = sent_tokenize(sentences)
                    sentences = [word_tokenize(s) for s in sentences]
                    sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
                    all_tokens.extend(sentences)
            except Exception as e:
                print(f'Could not read {f.name}: {e}')
    
    return all_tokens

In [205]:
enron_tokens = create_enron_tokens('data/enronsent')

Could not read .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte


In [208]:
sum([len(s) for s in enron_tokens])

13975435

In [ ]:
enron_model = Word2Vec(
    sentences=enron_tokens,
    vector_size=100,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    epochs=5,
    negative=5
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [212]:
enron_dist = FreqDist([t for line in enron_tokens for t in line])

In [214]:
enron_model.wv['build']

array([ 0.05960566, -0.13967216, -0.192964  , -0.34073707, -0.41870674,
       -0.23093434,  0.04985664,  0.49927938,  0.22975378, -0.41612378,
       -0.35994232, -0.6958026 ,  0.25747493,  0.01449603,  0.15946466,
       -0.1480282 , -0.29589376,  0.28780258, -0.18330908, -0.12624937,
       -0.10997437,  0.20908403,  0.0865448 , -0.577253  , -0.05241356,
       -0.05720013, -0.5927117 , -0.3148318 , -0.40854663,  0.00971841,
       -0.1797996 , -0.23060045,  0.20519073,  0.32311594, -0.22035463,
        0.09631245, -0.15192527, -0.14321151, -0.4104353 ,  0.1916394 ,
       -0.1985985 ,  0.13883348, -0.52788544, -0.5215233 ,  0.75597674,
        0.10496998, -0.48201478, -0.3537324 , -0.27254993,  0.13361508,
        0.24108045, -0.13600607, -0.07336839, -0.11009263,  0.15807141,
        0.3134331 ,  0.19610141,  0.03857709, -0.42260092,  0.38065958,
        0.66182375, -0.39915597,  0.11134535, -0.18690829, -0.3205693 ,
        0.03947898,  0.84460425,  0.3888131 , -0.17954965,  0.11

In [215]:
enron_model.wv.most_similar('build', topn=10)

[('develop', 0.7698425650596619),
 ('hinder', 0.702917754650116),
 ('spur', 0.6841839551925659),
 ('construct', 0.6794637441635132),
 ('maximise', 0.6786988973617554),
 ('modular', 0.6777182221412659),
 ('develope', 0.67405104637146),
 ('dynamically', 0.6730613112449646),
 ('forge', 0.6713972091674805),
 ('frustrate', 0.6711179614067078)]

In [216]:
enron_model.wv.most_similar('leverage', topn=10)

[('monetize', 0.7198493480682373),
 ('strengthen', 0.7019568085670471),
 ('optimize', 0.6980121731758118),
 ('productivity', 0.6946735978126526),
 ('complexities', 0.6944823861122131),
 ('teamwork', 0.6895453333854675),
 ('restructure', 0.6875494122505188),
 ('leveraging', 0.6845986843109131),
 ('navigate', 0.6827280521392822),
 ('disparate', 0.6818740963935852)]

In [217]:
coca_model.wv.most_similar('leverage', topn=10)

[('reshape', 0.7400580048561096),
 ('restructure', 0.7272158861160278),
 ('siphon', 0.7158498764038086),
 ('liquidity', 0.7049255967140198),
 ('extrapolate', 0.7009909152984619),
 ('garner', 0.6995500326156616),
 ('allocate', 0.697380781173706),
 ('stabilize', 0.6961700320243835),
 ('know-how', 0.6946137547492981),
 ('innovate', 0.6941611170768738)]